In [ ]:
from pathlib import Path
import pickle
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.tqdm

from python.config import load_config

cfg = load_config(os.environ.get('CONFIG', 'config.json'))

import python.utils as utils
from python.error_metrics import MergedErrorMetrics, LoadedErrorMetrics, FilteredErrorMetrics, MetricsIndex as MI # Alias
from python.forecast_reconciliation import reconcile_calculate_serialize_error_metrics
from python.sampled_hierarchies import SampledHierarchies, SingleStepHierarchies
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS
from python.aggregation import Aggregation
from python.base_forecasting import MODEL_MIN_TRAIN_OBS

# Setup

In [ ]:
FORECAST_CHECKPOINT = Path(f'sample/{cfg.FOLDER}/all_possible-{cfg.DATASET.START}-{cfg.DATASET.END}.pkl')
CAPTION_SUFFIX = f"{cfg.DATASET.NAME} dataset - {cfg.MODEL} model"
LABEL_SUFFIX = f"{cfg.DATASET.NAME}_{cfg.MODEL}"

Path(f"sample/{cfg.FOLDER}").mkdir(parents=True, exist_ok=True)
Path(f"metrics/{cfg.FOLDER}").mkdir(parents=True, exist_ok=True)

MIN_SERIES_LENGTH = cfg.HORIZON + MODEL_MIN_TRAIN_OBS[cfg.MODEL] * cfg.LEVELS.BASE_PERIOD


In [ ]:
AVAILABLE_LEVELS = Aggregation.complete_summing_matrix(n=cfg.LEVELS.SUBSET, seed=cfg.SEED)

# Forecasts and errors

In [ ]:
if cfg.DATASET.NAME == 'web-traffic':
    X_raw = utils.fetch_dataset_web_traffic()
    X = utils.filter_last_days_available(X_raw, 28*100)
if cfg.DATASET.NAME == 'M5':
    X = utils.fetch_dataset_m5()
if cfg.DATASET.NAME == 'M4':
    X = utils.fetch_dataset_m4()

X = utils.filter_last_days_available(X, cfg.HORIZON)
X = utils.reduce_dataset(X, start=cfg.DATASET.START, end=cfg.DATASET.END)
X = utils.align_fill_and_trim_series(X)
X = utils.filter_min_length(X, MIN_SERIES_LENGTH)

In [ ]:
if cfg.is_single_step_hierarchies():
    sh = SingleStepHierarchies(AVAILABLE_LEVELS)
else:
    sh = SampledHierarchies(AVAILABLE_LEVELS, cfg.LEVELS.DIFF, cfg.LEVELS.SEQUENCES, max_h=cfg.LEVELS.MAX_H)

all_possible = sh.full_hierarchy
minimal = sh.base_hierarchy

In [ ]:
all_possible.resample_time_series(X)

for cp in all_possible.make_base_forecasts(cfg.HORIZON, model=cfg.MODEL, n_jobs=cfg.N_JOBS, verbose=cfg.VERBOSE):
    with open(FORECAST_CHECKPOINT, 'wb') as f:
        pickle.dump(cp, f)

In [ ]:
with open(FORECAST_CHECKPOINT, 'rb') as f:
    all_possible = pickle.load(f)

for h in sh.hierarchies + [minimal]:
    h.copy_data(all_possible)

In [ ]:
reconcile_calculate_serialize_error_metrics(sh.hierarchies + [minimal],
                                            lambda h: f'sample/{cfg.FOLDER}/metrics-{h.name}-{cfg.DATASET.START}-{cfg.DATASET.END}.csv',
                                            verbose=cfg.VERBOSE,
                                            n_jobs=cfg.N_JOBS)

In [ ]:
ems = []
for file in Path(f"sample/{cfg.FOLDER}").rglob(f"*-{cfg.DATASET.START}-{cfg.DATASET.END}.csv"):
    if file.is_file():
        ems.append(LoadedErrorMetrics(file))

em = MergedErrorMetrics(ems)

em.save(f'metrics/{cfg.FOLDER}/sample_hiers_errors_raw-{cfg.DATASET.START}-{cfg.DATASET.END}.csv')

In [ ]:
em = LoadedErrorMetrics(f'metrics/{cfg.FOLDER}/sample_hiers_errors_raw-{cfg.DATASET.START}-{cfg.DATASET.END}.csv')

em = MergedErrorMetrics([
    FilteredErrorMetrics(em, MI(forecast_type=RECONCILED_FORECASTS)),
    FilteredErrorMetrics(em, MI(forecast_type=BASE_FORECASTS, hierarchy='Minimal'))
])

em.save(f'metrics/{cfg.FOLDER}/sample_hiers_errors.csv')